In [1]:
import pandas as pd
import numpy as np

TARGETS = ["Theta"]

SETS = [
    "Train_1",
    "Train_2",
    "Val",
    "Test_1",
    "Test_2",
    "Test_3"
]

results = pd.read_excel("resultados.xlsx")


In [2]:
best_models_tables = {}

summary_all = []     # estatísticas com todos
summary_top10 = []   # estatísticas com top 10

N = 10

for target in TARGETS:
    for dataset in SETS:
        
        col_r2 = f"R2_{dataset}_{target}"
        col_mse = f"MSE_{dataset}_{target}"
        
        if col_r2 in results.columns:
            
            table = results[
                ["model", "Neurons", col_r2, col_mse]
            ].sort_values(by=col_r2, ascending=False)
            
            # 🔹 Remove colchetes
            for col in [col_r2, col_mse]:
                table[col] = (
                    table[col]
                    .astype(str)
                    .str.strip("[]")
                    .astype(float)
                )
            
            best_models_tables[f"{dataset}_{target}"] = table
            
            # ===============================
            # 🔹 Estatísticas - TODOS
            # ===============================
            summary_all.append({
                "dataset": dataset,
                "target": target,
                "mean_r2": table[col_r2].mean(),
                "std_r2": table[col_r2].std(),
                "mean_mse": table[col_mse].mean(),
                "std_mse": table[col_mse].std()
            })
            
            # ===============================
            # 🔹 Estatísticas - TOP 10
            # ===============================
            top10 = table.head(N)
            
            summary_top10.append({
                "dataset": dataset,
                "target": target,
                "mean_r2": top10[col_r2].mean(),
                "std_r2": top10[col_r2].std(),
                "mean_mse": top10[col_mse].mean(),
                "std_mse": top10[col_mse].std()
            })


# 🔹 DataFrames finais
df_summary_all = pd.DataFrame(summary_all)
df_summary_top10 = pd.DataFrame(summary_top10)


# 🔹 Exportar para duas abas
with pd.ExcelWriter("Resumo_Estatisticas.xlsx") as writer:
    df_summary_all.to_excel(writer, sheet_name="Todos_Modelos", index=False)
    df_summary_top10.to_excel(writer, sheet_name="Top_10_Modelos", index=False)

print("Arquivo exportado com duas abas com sucesso.")


Arquivo exportado com duas abas com sucesso.


In [4]:
df_summary_top10

,dataset,target,mean_r2,std_r2,mean_mse,std_mse
0,Train_1,Theta,0.899222,0.008939,0.022935,0.002034
1,Train_2,Theta,0.831854,0.003668,0.038386,0.000837
2,Val,Theta,0.662914,0.051766,0.110036,0.016898
3,Test_1,Theta,0.900989,0.034097,0.022681,0.007811
4,Test_2,Theta,0.892674,0.007003,0.025450,0.001661
5,Test_3,Theta,0.761908,0.005797,0.078098,0.001901


In [3]:
best_only = []
for dataset in SETS:
    for target in TARGETS:
        col_r2 = f"R2_{dataset}_{target}"
        
        if col_r2 in results.columns:
            
            idx_best = results[col_r2].idxmax()
            
            best_only.append({
                "Target": target,
                "Dataset": dataset,
                "Best_Model": results.loc[idx_best, "model"],
                "Neurons": results.loc[idx_best, "Neurons"],
                "Best_R2": results.loc[idx_best, col_r2]
            })

best_only_df = pd.DataFrame(best_only)

print("\n===== MELHOR MODELO POR TARGET/DATASET =====")
print(best_only_df)



===== MELHOR MODELO POR TARGET/DATASET =====
  Target  Dataset                                       Best_Model  \
0  Theta  Train_1          model_arch8-4_r0.9_Ld0.3_Lp0.7_seed5960   
1  Theta  Train_2           model_arch64_r0.9_Ld0.7_Lp0.3_seed5830   
2  Theta      Val         model_arch264_r0.01_Ld0.3_Lp0.7_seed5960   
3  Theta   Test_1          model_arch8-4_r0.9_Ld0.3_Lp0.7_seed5960   
4  Theta   Test_2  model_arch264-128-64_r0.01_Ld0.3_Lp0.7_seed5830   
5  Theta   Test_3         model_arch264_r0.01_Ld0.3_Lp0.7_seed2072   

          Neurons   Best_R2  
0          [8, 4]  0.918626  
1            [64]  0.838942  
2           [264]  0.744077  
3          [8, 4]  0.944030  
4  [264, 128, 64]  0.910172  
5           [264]  0.770310  
